In [119]:
import requests
from bs4 import BeautifulSoup
import urllib.parse
import json

def crawl_job_data(keyword):
    encoded_keyword = urllib.parse.quote(keyword)
    url = f"https://www.jobkorea.co.kr/Search/?stext={encoded_keyword}&tabType=recruit&Page_No=1"
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.text, "html.parser")
    jobs = soup.select('article.list-item')
    job_list = []
    for job in jobs[:20]:
        try:
            company_names = job.select_one('a.corp-name-link').text.strip()
            job_title_element = job.select_one('a.information-title-link.dev-view')
            if job_title_element:
                job_title = job_title_element.text.strip()
                link = urllib.parse.urljoin("https://www.jobkorea.co.kr", job_title_element['href'])
                full_link = urllib.parse.urljoin("https://www.jobkorea.co.kr", link)
                full_link = urllib.parse.quote(full_link, safe=':/&=?')
            else:
                job_title = "제목 없음"
                link = None
            job_info_list = []
            job_info_ul = job.select_one('ul.chip-information-group')
            if job_info_ul:
                job_info_items = job_info_ul.find_all('li', class_='chip-information-item')
                job_info_list = [item.text.strip() for item in job_info_items]
            job_details = {}
            # 상세페이지 크롤
            if link:
                response_detail = requests.get(link, headers={'User-Agent': 'Mozilla/5.0'})
                response_detail.raise_for_status()
                soup_detail = BeautifulSoup(response_detail.text, "html.parser")
                qualification_info = {tag.text.strip(): value.text.strip().replace('\n', '').replace('\r', '').replace('\t', '').replace(' ', '') for tag, value in zip(soup_detail.select('div.tbCol')[0].select('dt'), soup_detail.select('div.tbCol')[0].select('dd'))}
                employment_info = {tag.text.strip(): value.text.strip().replace('\n', '').replace('\r', '').replace('\t', '').replace(' ', '') for tag, value in zip(soup_detail.select('div.tbCol')[1].select('dt'), soup_detail.select('div.tbCol')[1].select('dd'))}
                company_info = {tag.text.strip(): value.text.strip().replace('\n', '').replace('\r', '').replace('\t', '').replace(' ', '') for tag, value in zip(soup_detail.select('div.tbCol')[2].select('dt'), soup_detail.select('div.tbCol')[2].select('dd'))}

                # 이미지 src 추출 및 JSON 형태로 추가
                img_tag = soup_detail.find('img', id='cologo')
                if img_tag and 'src' in img_tag.attrs:
                    src_value = img_tag['src'].strip()
                    company_info["company_img_src"] = src_value
                else:
                    company_info["company_img_src"] = "이미지 없음"

                job_details = {
                    "qualification": qualification_info,
                    "employment": employment_info,
                    "company": company_info,
                }
            job_data = {
                "company_name": company_names,
                "title": job_title,
                "link": full_link,
                "info": job_info_list,
                "details": job_details
            }
            job_list.append(job_data)
        except Exception as e:
            print(f"Error processing job: {e}")
    return job_list

keyword = '데이터 분석'
job_data_list = crawl_job_data(keyword)
json_result = json.dumps(job_data_list, ensure_ascii=False, indent=4)
print(json_result)

Error processing job: list index out of range
[
    {
        "company_name": "㈜인라이플",
        "title": "[주4.5일근무] 데이터 분석 및 기획자 채용",
        "link": "https://www.jobkorea.co.kr/Recruit/GI_Read/46607234?Oem_Code=C1&logpath=1&stext=%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EB%B6%84%EC%84%9D&listno=1",
        "info": [
            "경력2년↑",
            "학력무관",
            "정규직",
            "서울 구로구 외",
            "D-49"
        ],
        "details": {
            "qualification": {
                "경력": "경력(2년이상)",
                "학력": "학력무관",
                "스킬": "API,JAVA,JSP,MongoDB,MySQL,Python,Pytorch,R,Linux"
            },
            "employment": {
                "고용형태": "정규직수습3개월",
                "급여": "회사내규에따름-면접후결정",
                "지역": "서울시구로구,관악구,금천구,영등포구,동작구지도",
                "시간": "주4.5일(매주금요일1시퇴근)10:00~19:00"
            },
            "company": {
                "산업(업종)": "솔루션·SI·CRM·ERP",
                "사원수": "300명",
                "설립년도": "2012년(14년차)",
             